# Chapter 16: Data Pipeline — Collecting, Cleaning, Preparing Data

Models are only as good as their data. The saying in ML:

```
"Garbage in, garbage out"
```

Most ML projects spend **80% of time on data** and 20% on the model.

This chapter covers the full journey:
```
Raw data → Clean → Transform → Split → Train
```

---
## The Data Pipeline Overview

```
┌──────────┐    ┌──────────┐    ┌───────────┐    ┌─────────┐    ┌───────┐
│ Collect  │ →  │  Clean   │ →  │ Transform │ →  │  Split  │ →  │ Train │
│          │    │          │    │           │    │         │    │       │
│ scrape   │    │ missing  │    │ normalize │    │ train   │    │ model │
│ APIs     │    │ dupes    │    │ encode    │    │ val     │    │       │
│ databases│    │ outliers │    │ tokenize  │    │ test    │    │       │
└──────────┘    └──────────┘    └───────────┘    └─────────┘    └───────┘
```

In [ ]:
import numpy as np
np.random.seed(42)

# Simulate a messy real-world dataset
print("Real-World Data is MESSY:\n")

# Customer data with problems
raw_data = [
    {"name": "Alice",   "age": 29,   "income": 55000,  "city": "NYC"},
    {"name": "Bob",     "age": None,  "income": 72000,  "city": "nyc"},      # missing age, inconsistent city
    {"name": "Charlie", "age": 35,   "income": -5000,  "city": "New York"}, # negative income??
    {"name": "Alice",   "age": 29,   "income": 55000,  "city": "NYC"},      # duplicate!
    {"name": "Diana",   "age": 250,  "income": 68000,  "city": "LA"},       # age 250??
    {"name": "Eve",     "age": 31,   "income": 999999, "city": "SF"},       # outlier income
    {"name": "",        "age": 28,   "income": 45000,  "city": "LA"},       # empty name
    {"name": "Frank",   "age": 42,   "income": 61000,  "city": "sf"},       # inconsistent city
]

print(f"  Raw dataset: {len(raw_data)} rows\n")
print(f"  {'Name':<10} {'Age':<6} {'Income':<10} {'City'}")
print(f"  {'─'*10} {'─'*6} {'─'*10} {'─'*10}")
for row in raw_data:
    problems = []
    if row['age'] is None: problems.append("missing")
    elif row['age'] > 150: problems.append("invalid")
    if row['income'] < 0: problems.append("negative")
    if row['income'] > 500000: problems.append("outlier")
    if not row['name']: problems.append("empty")
    
    age_str = str(row['age']) if row['age'] is not None else 'None'
    flag = f"  ← {', '.join(problems)}" if problems else ""
    print(f"  {row['name']:<10} {age_str:<6} {row['income']:<10} {row['city']:<10}{flag}")

print(f"\n  Problems found: missing values, duplicates, outliers,")
print(f"  invalid values, inconsistent formats, empty fields")

---
## Step 1: Data Cleaning

Common cleaning tasks:
1. **Remove duplicates** — exact or near-duplicates
2. **Handle missing values** — drop, fill with mean/median, or interpolate
3. **Fix invalid values** — out-of-range, wrong types
4. **Standardize formats** — dates, categories, text case
5. **Remove outliers** — values that are likely errors

In [ ]:
# Data cleaning pipeline
print("Data Cleaning Pipeline:\n")

def clean_data(raw):
    steps = []
    data = raw.copy()
    
    # Step 1: Remove duplicates
    seen = set()
    deduped = []
    for row in data:
        key = (row['name'], row['age'], row['income'])
        if key not in seen:
            seen.add(key)
            deduped.append(row)
    removed = len(data) - len(deduped)
    steps.append(f"Remove duplicates: {removed} removed")
    data = deduped
    
    # Step 2: Remove rows with empty required fields
    before = len(data)
    data = [r for r in data if r['name']]
    steps.append(f"Remove empty names: {before - len(data)} removed")
    
    # Step 3: Handle missing values (fill age with median)
    valid_ages = [r['age'] for r in data if r['age'] is not None and r['age'] < 150]
    median_age = sorted(valid_ages)[len(valid_ages) // 2]
    filled = 0
    for row in data:
        if row['age'] is None:
            row['age'] = median_age
            filled += 1
    steps.append(f"Fill missing ages with median ({median_age}): {filled} filled")
    
    # Step 4: Fix invalid values
    fixed = 0
    for row in data:
        if row['age'] > 150:
            row['age'] = median_age  # replace impossible age
            fixed += 1
        if row['income'] < 0:
            row['income'] = 0  # can't have negative income
            fixed += 1
    steps.append(f"Fix invalid values: {fixed} corrected")
    
    # Step 5: Standardize city names
    city_map = {'nyc': 'NYC', 'new york': 'NYC', 'sf': 'SF', 'la': 'LA'}
    for row in data:
        row['city'] = city_map.get(row['city'].lower(), row['city'])
    steps.append("Standardize city names")
    
    # Step 6: Handle outliers (cap at 3 standard deviations)
    incomes = [r['income'] for r in data]
    mean_inc = np.mean(incomes)
    std_inc = np.std(incomes)
    cap = mean_inc + 3 * std_inc
    capped = 0
    for row in data:
        if row['income'] > cap:
            row['income'] = int(cap)
            capped += 1
    steps.append(f"Cap outlier incomes (>{int(cap)}): {capped} capped")
    
    return data, steps

cleaned, steps = clean_data(raw_data)

for i, step in enumerate(steps, 1):
    print(f"  Step {i}: {step}")

print(f"\n  Result: {len(raw_data)} rows → {len(cleaned)} clean rows\n")
print(f"  {'Name':<10} {'Age':<6} {'Income':<10} {'City'}")
print(f"  {'─'*10} {'─'*6} {'─'*10} {'─'*10}")
for row in cleaned:
    print(f"  {row['name']:<10} {row['age']:<6} {row['income']:<10} {row['city']}")

---
## Step 2: Feature Engineering

Transform raw data into features the model can use:

- **Numerical**: normalize/standardize to similar scales
- **Categorical**: one-hot encode ("NYC" → [1, 0, 0])
- **Text**: tokenize → embed (Ch 10-11)
- **Derived**: create new features from existing ones

In [ ]:
# Feature engineering
print("Feature Engineering:\n")

# Numerical: Normalization vs Standardization
ages = np.array([29, 35, 33, 31, 42])
incomes = np.array([55000, 72000, 0, 68000, 61000])

print("1. Numerical Features — Scaling\n")
print(f"   Raw ages:    {ages}")
print(f"   Raw incomes: {incomes}\n")

# Min-max normalization: scale to [0, 1]
ages_norm = (ages - ages.min()) / (ages.max() - ages.min())
incomes_norm = (incomes - incomes.min()) / (incomes.max() - incomes.min())

print(f"   Min-Max Normalized (0 to 1):")
print(f"     ages:    {ages_norm.round(3)}")
print(f"     incomes: {incomes_norm.round(3)}\n")

# Z-score standardization: mean=0, std=1
ages_std = (ages - ages.mean()) / ages.std()
incomes_std = (incomes - incomes.mean()) / incomes.std()

print(f"   Z-Score Standardized (mean=0, std=1):")
print(f"     ages:    {ages_std.round(3)}")
print(f"     incomes: {incomes_std.round(3)}\n")

print(f"   Why scale? Without it:")
print(f"     income=55000 dominates over age=29")
print(f"     Model thinks income is 1900x more important!")
print(f"     After scaling, both features are on equal footing.")

In [ ]:
# One-hot encoding for categorical features
print("2. Categorical Features — One-Hot Encoding\n")

cities = ["NYC", "NYC", "NYC", "LA", "SF"]
unique_cities = sorted(set(cities))

print(f"   Cities: {cities}")
print(f"   Unique: {unique_cities}\n")

# One-hot encode
print(f"   One-Hot Encoding:")
print(f"   {'City':<6}", end="")
for c in unique_cities:
    print(f" {c:>4}", end="")
print()
print(f"   {'─'*6}", end="")
print(f" {'─'*4}" * len(unique_cities))

for city in cities:
    encoding = [1 if city == c else 0 for c in unique_cities]
    print(f"   {city:<6}", end="")
    for v in encoding:
        print(f" {v:>4}", end="")
    print()

print(f"\n   Why not just use numbers (NYC=1, LA=2, SF=3)?")
print(f"   Because then the model thinks SF > LA > NYC (ordered relationship).")
print(f"   One-hot says: these are just different categories, no order.")

# Derived features
print(f"\n3. Derived Features — Create From Existing\n")
print(f"   Examples:")
print(f"     age + income → income_per_year_of_age")
print(f"     date_of_birth → age, zodiac_sign, generation")
print(f"     text length → num_words, avg_word_length")
print(f"     timestamp → hour_of_day, day_of_week, is_weekend")

---
## Step 3: Train/Validation/Test Split

**Critical rule**: never let the model see test data during training!

```
All Data (100%)
├── Train (70-80%)  → model learns from this
├── Validation (10-15%) → tune hyperparameters, early stopping
└── Test (10-15%)   → final evaluation (touch ONCE)
```

Why three splits?
- **Train**: model learns patterns
- **Validation**: catch overfitting, tune settings
- **Test**: unbiased final score (if you peek, it's contaminated)

In [ ]:
# Data splitting
print("Train / Validation / Test Split:\n")

# Generate sample data
n_samples = 1000
X = np.random.randn(n_samples, 5)
y = (X[:, 0] + X[:, 1] > 0).astype(int)  # simple classification

# Shuffle
indices = np.random.permutation(n_samples)
X, y = X[indices], y[indices]

# Split: 70% train, 15% val, 15% test
train_end = int(0.7 * n_samples)
val_end = int(0.85 * n_samples)

X_train, y_train = X[:train_end], y[:train_end]
X_val, y_val = X[train_end:val_end], y[train_end:val_end]
X_test, y_test = X[val_end:], y[val_end:]

print(f"  Total samples: {n_samples}")
print(f"  ┌─────────────────────────────────────────────────────────┐")
print(f"  │{'█' * 35}{'░' * 8}{'·' * 8}                │")
print(f"  │{'Train (70%)':^35}{'Val':^8}{'Test':^8}                │")
print(f"  └─────────────────────────────────────────────────────────┘")
print(f"  Train:      {len(X_train):>4} samples ({len(X_train)/n_samples*100:.0f}%)")
print(f"  Validation: {len(X_val):>4} samples ({len(X_val)/n_samples*100:.0f}%)")
print(f"  Test:       {len(X_test):>4} samples ({len(X_test)/n_samples*100:.0f}%)")

# Check class balance in each split
print(f"\n  Class balance (% positive):")
print(f"    Train: {y_train.mean()*100:.1f}%")
print(f"    Val:   {y_val.mean()*100:.1f}%")
print(f"    Test:  {y_test.mean()*100:.1f}%")
print(f"  (Should be similar — shuffle ensures this)")

In [ ]:
# Why splitting matters — overfitting demo
print("Why Splitting Matters — Overfitting Demo:\n")

# Simple model: weighted sum
def train_model(X, y, epochs=100, lr=0.1):
    w = np.random.randn(X.shape[1]) * 0.1
    for _ in range(epochs):
        pred = 1 / (1 + np.exp(-X @ w))  # sigmoid
        grad = X.T @ (pred - y) / len(y)
        w -= lr * grad
    return w

def accuracy(X, y, w):
    pred = (1 / (1 + np.exp(-X @ w))) > 0.5
    return np.mean(pred == y)

# Train on training set
w = train_model(X_train, y_train, epochs=200)

train_acc = accuracy(X_train, y_train, w)
val_acc = accuracy(X_val, y_val, w)
test_acc = accuracy(X_test, y_test, w)

print(f"  Model trained on {len(X_train)} samples:\n")
print(f"    Train accuracy: {train_acc:.3f}  (model saw this data)")
print(f"    Val accuracy:   {val_acc:.3f}  (used to tune hyperparams)")
print(f"    Test accuracy:  {test_acc:.3f}  (final unbiased score)\n")

print(f"  If train >> val/test → model is OVERFITTING")
print(f"  (memorizing training data instead of learning patterns)\n")

# Now show what happens with tiny training set (extreme overfit)
w_tiny = train_model(X_train[:20], y_train[:20], epochs=500)
tiny_train_acc = accuracy(X_train[:20], y_train[:20], w_tiny)
tiny_test_acc = accuracy(X_test, y_test, w_tiny)

print(f"  With only 20 training samples (overfit scenario):")
print(f"    Train accuracy: {tiny_train_acc:.3f}  ← looks great!")
print(f"    Test accuracy:  {tiny_test_acc:.3f}  ← reality check")

---
## Data for LLMs: Pre-training vs Fine-tuning

| Stage | Data Source | Size | Format |
|-------|-----------|------|--------|
| Pre-training | Web crawl, books, code | 1-15 trillion tokens | Raw text |
| SFT | Human-written responses | 10K-100K examples | (instruction, response) |
| RLHF | Human preferences | 1K-50K pairs | (prompt, chosen, rejected) |
| LoRA | Domain-specific text | 1K-10K examples | Task-specific format |

In [ ]:
# LLM pre-training data pipeline
print("LLM Pre-Training Data Pipeline:\n")

print("  ┌─────────────────────────────────────────────────────────────┐")
print("  │ Step 1: CRAWL                                              │")
print("  │   Source: Common Crawl (billions of web pages)             │")
print("  │   Size:  ~100 TB of raw HTML                               │")
print("  └─────────────────────────┬───────────────────────────────────┘")
print("                            ↓")
print("  ┌─────────────────────────────────────────────────────────────┐")
print("  │ Step 2: EXTRACT TEXT                                        │")
print("  │   Remove HTML, navigation, ads, boilerplate                │")
print("  │   Keep: article text, code, documentation                  │")
print("  └─────────────────────────┬───────────────────────────────────┘")
print("                            ↓")
print("  ┌─────────────────────────────────────────────────────────────┐")
print("  │ Step 3: QUALITY FILTER                                      │")
print("  │   Remove: spam, porn, duplicates, low-quality text         │")
print("  │   Keep:  well-written, informative, diverse content        │")
print("  │   Tools: perplexity filter, classifier, dedup (MinHash)    │")
print("  └─────────────────────────┬───────────────────────────────────┘")
print("                            ↓")
print("  ┌─────────────────────────────────────────────────────────────┐")
print("  │ Step 4: DECONTAMINATION                                     │")
print("  │   Remove test/benchmark data from training set!            │")
print("  │   If model saw the test → cheating → scores are fake       │")
print("  └─────────────────────────┬───────────────────────────────────┘")
print("                            ↓")
print("  ┌─────────────────────────────────────────────────────────────┐")
print("  │ Step 5: TOKENIZE & PACK                                     │")
print("  │   BPE tokenize (Ch 10), pack into fixed-length sequences   │")
print("  │   Result: billions of token sequences ready for training   │")
print("  └─────────────────────────────────────────────────────────────┘")

print("\n  Data reduction at each step:")
print("    Raw crawl:      100 TB")
print("    After extract:   20 TB")
print("    After filter:     5 TB")
print("    After dedup:      2 TB")
print("    Final tokens:  1-2 trillion tokens")

In [ ]:
# Deduplication — why it matters
print("Deduplication: Why It Matters\n")

# Simulate training data with duplicates
print("  If 'Hello world' appears 10,000 times in training data:")
print("  → Model memorizes it, generates it too often")
print("  → Wasted compute (learning same thing 10,000 times)\n")

# Show dedup techniques
print("  Deduplication Methods:\n")
print("  1. Exact dedup:  hash entire documents")
print("     'Hello world' == 'Hello world' → remove one\n")
print("  2. Near dedup (MinHash):  fuzzy matching")
print("     'The cat sat on mat' ≈ 'The cat sat on the mat' → remove one\n")
print("  3. N-gram dedup:  remove docs sharing long passages")
print("     If 50+ consecutive tokens match → likely duplicate\n")

# Simple hash-based dedup demo
documents = [
    "The quick brown fox jumps over the lazy dog",
    "Machine learning is a subset of AI",
    "The quick brown fox jumps over the lazy dog",  # exact dup
    "Python is a programming language",
    "machine learning is a subset of AI",  # near dup (case)
    "Python is a programming language",  # exact dup
]

print(f"  Demo — Exact dedup with hashing:")
seen_hashes = set()
unique_docs = []
for doc in documents:
    h = hash(doc.lower())  # case-insensitive
    if h not in seen_hashes:
        seen_hashes.add(h)
        unique_docs.append(doc)
        print(f"    KEEP: '{doc[:50]}'")
    else:
        print(f"    DUP:  '{doc[:50]}'")

print(f"\n  {len(documents)} docs → {len(unique_docs)} unique ({len(documents)-len(unique_docs)} removed)")

---
## Data Quality > Data Quantity

Recent research shows: **less but higher quality data** often beats more noisy data.

```
Phi-1.5 (Microsoft): 1.3B params trained on "textbook quality" data
  → Matches models 10x larger trained on noisy web data!
```

Quality signals:
- Well-structured text (proper grammar, coherent arguments)
- Educational content (textbooks, documentation)
- Diverse topics (not 90% social media)
- Correct information (not misinformation)

In [ ]:
# Quality vs Quantity experiment
print("Quality vs Quantity — A Simple Experiment:\n")

# True pattern: y = 2*x + 1
def true_function(x):
    return 2 * x + 1

# High quality data: clean, minimal noise
x_high = np.random.randn(50)
y_high = true_function(x_high) + np.random.randn(50) * 0.1  # low noise

# Low quality data: noisy, some mislabeled
x_low = np.random.randn(500)  # 10x more data!
y_low = true_function(x_low) + np.random.randn(500) * 2.0  # high noise
# Add some mislabeled points (data errors)
corrupt_idx = np.random.choice(500, 50, replace=False)
y_low[corrupt_idx] = np.random.randn(50) * 10  # garbage values

# Train simple linear regression on both
def fit_linear(x, y):
    # y = wx + b, solve with least squares
    X = np.column_stack([x, np.ones_like(x)])
    w = np.linalg.lstsq(X, y, rcond=None)[0]
    return w

w_high = fit_linear(x_high, y_high)
w_low = fit_linear(x_low, y_low)

# Test on clean data
x_test = np.random.randn(100)
y_test = true_function(x_test)

pred_high = np.column_stack([x_test, np.ones(100)]) @ w_high
pred_low = np.column_stack([x_test, np.ones(100)]) @ w_low

error_high = np.mean((pred_high - y_test) ** 2)
error_low = np.mean((pred_low - y_test) ** 2)

print(f"  True function: y = 2x + 1\n")
print(f"  High-quality data (50 samples, low noise):")
print(f"    Learned: y = {w_high[0]:.3f}x + {w_high[1]:.3f}")
print(f"    Test error: {error_high:.4f}\n")

print(f"  Low-quality data (500 samples, high noise + errors):")
print(f"    Learned: y = {w_low[0]:.3f}x + {w_low[1]:.3f}")
print(f"    Test error: {error_low:.4f}\n")

winner = "HIGH quality" if error_high < error_low else "LOW quality"
print(f"  Winner: {winner} (even with 10x less data!)")
print(f"  This is why data cleaning matters so much.")

---
## Data Augmentation

When you don't have enough data, **create more** from what you have:

```
Images:  flip, rotate, crop, change brightness
Text:    synonym replacement, back-translation, paraphrase
Audio:   add noise, change speed, shift pitch
```

In [ ]:
# Text augmentation techniques
print("Data Augmentation for Text:\n")

original = "The cat quickly jumped over the fence"
print(f"  Original: '{original}'\n")

# Technique 1: Synonym replacement
synonyms = {"quickly": "rapidly", "jumped": "leaped", "fence": "barrier"}
augmented1 = original
for word, syn in synonyms.items():
    augmented1 = augmented1.replace(word, syn)
print(f"  1. Synonym replacement:")
print(f"     '{augmented1}'\n")

# Technique 2: Random deletion
words = original.split()
delete_idx = 2  # remove "quickly"
augmented2 = ' '.join(words[:delete_idx] + words[delete_idx+1:])
print(f"  2. Random word deletion:")
print(f"     '{augmented2}'\n")

# Technique 3: Random swap
words = original.split()
words[2], words[3] = words[3], words[2]  # swap positions
augmented3 = ' '.join(words)
print(f"  3. Random word swap:")
print(f"     '{augmented3}'\n")

# Technique 4: Back-translation (simulate)
print(f"  4. Back-translation (English → French → English):")
print(f"     'The cat rapidly leapt above the fence'\n")

print(f"  From 1 example → 4+ training examples!")
print(f"  The label stays the same, but the model sees variation.")

print(f"\n\n  For LLM fine-tuning, augmentation options:")
print(f"    • Rephrase instructions (same meaning, different wording)")
print(f"    • Use LLM to generate variations of training examples")
print(f"    • Translate to another language and back")
print(f"    • Add/remove context around the core question")

---
## Handling Imbalanced Data

When one class is much rarer than others:

```
Fraud detection:  99.9% normal, 0.1% fraud
Disease diagnosis: 99% healthy, 1% sick
```

A model that always predicts "normal" gets 99.9% accuracy but is useless!

In [ ]:
# Class imbalance problem
print("Class Imbalance Problem:\n")

# Simulate imbalanced fraud dataset
n_normal = 9900
n_fraud = 100
total = n_normal + n_fraud

print(f"  Fraud Detection Dataset:")
print(f"    Normal transactions: {n_normal:>5} ({n_normal/total*100:.1f}%)")
print(f"    Fraud transactions:  {n_fraud:>5} ({n_fraud/total*100:.1f}%)")
print(f"    Total:               {total:>5}\n")

print(f"  'Always predict normal' accuracy: {n_normal/total*100:.1f}% ← misleading!\n")

print(f"  Solutions:\n")
print(f"  1. Oversample minority class (duplicate fraud examples):")
print(f"     Normal: {n_normal}  Fraud: {n_normal} (duplicated 99x)\n")

print(f"  2. Undersample majority class (use fewer normal):")
print(f"     Normal: {n_fraud*2}  Fraud: {n_fraud}\n")

print(f"  3. Class weights (penalize missing fraud more):")
print(f"     loss = 1.0 × normal_errors + 99.0 × fraud_errors\n")

print(f"  4. SMOTE (generate synthetic minority examples):")
print(f"     Create new fraud examples by interpolating between existing ones\n")

# Show class weighting
print(f"  Class Weighting Example:")
print(f"    Without weights: model ignores rare class (easy 99% accuracy)")
print(f"    With weights:    missing 1 fraud costs as much as missing 99 normals")
print(f"    → Forces model to learn fraud patterns even though they're rare")

---
## Data Versioning and Reproducibility

ML experiments must be **reproducible**:

```
Same data + Same code + Same seed = Same result
```

Tools:
- **Random seeds** — deterministic shuffling and splitting
- **Data versioning** (DVC) — track data changes like git tracks code
- **Metadata logging** — record what data was used for each experiment

In [ ]:
# Reproducibility
print("Reproducibility: Same Data → Same Result\n")

# Without seed: different results every time
results_no_seed = []
for trial in range(5):
    data = np.random.randn(100)
    results_no_seed.append(data.mean())

print(f"  Without fixed seed (different each run):")
for i, r in enumerate(results_no_seed):
    print(f"    Trial {i+1}: mean = {r:.4f}")

# With seed: same results every time
results_with_seed = []
for trial in range(5):
    np.random.seed(42)  # reset seed each time
    data = np.random.randn(100)
    results_with_seed.append(data.mean())

print(f"\n  With fixed seed (same every run):")
for i, r in enumerate(results_with_seed):
    print(f"    Trial {i+1}: mean = {r:.4f}  {'← all identical!' if i > 0 else ''}")

print(f"\n  Best practices:")
print(f"    • Set random seed at start of every experiment")
print(f"    • Log: seed, data version, preprocessing steps, hyperparameters")
print(f"    • Use data versioning (DVC) to track dataset changes")
print(f"    • Never modify data in-place — create new versions")

---
## Summary

| Concept | Key Point |
|---------|----------|
| **Data cleaning** | Remove duplicates, fix invalid values, standardize formats |
| **Feature engineering** | Normalize numbers, one-hot encode categories |
| **Train/val/test split** | Never let model see test data during training |
| **LLM data pipeline** | Crawl → extract → filter → dedup → tokenize |
| **Quality > quantity** | Clean small dataset often beats noisy large one |
| **Augmentation** | Create more data from existing examples |
| **Class imbalance** | Use oversampling, class weights, or SMOTE |
| **Reproducibility** | Seeds + versioning = same results every time |

**Next chapter**: Evaluation — how to measure if your model is actually good